# Checking Value Labels

The purpose here is to understand whether individaul months' data can be loaded to a single relational database table or must be kept separate or transformed first before loading.
Obviously post-processing will be much easier if it can be kept to a single table.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import pymongo
from collections import Counter
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

Retrieve value labels from MongoDB database.
Because the 2023 rebase likely differs systematically from the previous rebases, check the 2023 rebase version first.
If the differences are too substantial, it can be okay to just run analyses on 2006 onward data.
This is all for fun only anyway.

In [2]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_var_labs = lfs_db['variable_labels']
varlabs = list(lfs_var_labs.find(filter={'file': {'$regex': r'(?i:rebase2023)'}}, projection={'_id': 0}))
lfs_val_labs = lfs_db['value_labels']
vallabs = list(lfs_val_labs.find(filter={'file': {'$regex': r'(?i:rebase2023)'}}, projection={'_id': 0}))
client.close()

Pinged your deployment. You successfully connected to MongoDB!


First check whether the same variables appear in all survey years.

In [3]:
varlabs = {vl['file']: vl['labels'] for vl in varlabs}
varlabs_count = Counter(v for _, vl in varlabs.items() for v, _ in vl.items())
Counter(c for v, c in varlabs_count.items())

Counter({17: 60})

That means all 60 variables appear in all 17 syntax files with the exact same variable names.
Since that is the case, next is to check whether all value labels align.

In [4]:
vallabs = {vl['file']: {v: {i: s for s, i in l.items()} for v, l in vl['labels'].items()} for vl in vallabs}
vallabs_vars = Counter(v for _, vl in vallabs.items() for v, _ in vl.items())
vallabs = {v: {f: vl[v] for f, vl in vallabs.items()} for v in vallabs_vars.keys()}
len(vallabs.keys())

43

There are 43 variables to check.
This is going to be somewhat lengthy.
To get around this we are going to use the following trick.
If all years' value labels agree, then treating this as an inter-rater reliability exercise will yield a perfect inter-rater reliability score.
This means that we can quickly iterate through all categorical variables and only highlight those variables with categories that do not agree and print them out for further inspection.

In [5]:
def check_value_label_agreement(data):
    assert isinstance(data, pd.DataFrame), 'Function assumes data is passed as a pandas DataFrame.'
    ratings = aggregate_raters(data)
    kappa = fleiss_kappa(ratings[0])
    return {'kappa': kappa, 'ratings': ratings[0], 'columns': ratings[1]}

Test function output on one variable.

In [34]:
check_value_label_agreement(pd.DataFrame.from_dict(vallabs[next(iter(vallabs))]).fillna('Missing'))

{'kappa': 1.0,
 'ratings': array([[ 0,  0,  0,  0, 17,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 17,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 17,  0,  0,  0,  0],
        [17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0, 17,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0, 17,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0, 17,  0,  0,  0,  0,  0,  0],
        [ 0, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 17],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 17,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0, 17,  0,  0],
        [ 0,  0, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0]]),
 'columns': array(['April', 'August', 'December', 'February', 'January', 'July',
        'June', 'March', 'May', 'November', 'October', 'September'],
       dtype=object)}

Check outputs for any variable without perfect agreement.

In [35]:
for v, vl in vallabs.items():
    rating = check_value_label_agreement(pd.DataFrame.from_dict(vl).fillna('Missing'))
    if rating['kappa'] != 1:
        print(v)
        print(rating, '\n')

LKPUBAG
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKEMPLOY
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKRELS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKATADS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKANSADS
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 

LKOTHERN
{'kappa': nan, 'ratings': array([[17]]), 'columns': array(['Yes'], dtype=object)} 



/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py:267: RuntimeWarning: invalid value encountered in scalar divide
  kappa = (p_mean - p_mean_exp) / (1- p_mean_exp)
/home/gohyc/miniconda3/envs/basic-data-science/lib/python3.10/site-packages/statsmodels/stats/inter_rater.py

Value labels have perfect agreement.
We can now also quickly check if there were changes in variable descriptions.

In [39]:
check_value_label_agreement(pd.DataFrame.from_dict(varlabs).fillna('Missing'))

{'kappa': 1.0,
 'ratings': array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]),
 'columns': array(['Actual hours worked per week at all jobs',
        'Actual hours worked per week at main job',
        'Age in 2 and 3 year groups, 15 to 29', 'Age of youngest child',
        'Availability during the reference week',
        'Class of worker, main job', 'Current student status',
        'Duration of joblessness (months)',
        'Duration of unemployment (weeks)', 'Establishment size',
        'Firm size', 'Five-year age group of respondent',
        'Flows into unemployment',
        'Full- or part-time status at main or only job',
        'Full- or part-time status of last job',
        'Highest educational attainment',
        'Hours away from work, part-week absence only', 'Immigrant status',
        'Industry of main job', '

# Summary

All value labels and variable descriptions from 2006-2022 have perfect agreement.
The basic implication is that with a bit of clean-up where necessary, all data for 2006-2023 can be uploaded to the same data table in a relational database.

Pre-2006 variables will be examined in due time, but it is not necessary for now.
Jan 2006 to Apr 2023 is 208 months of data.